[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/cyclides/cyclides.ipynb)

# Dupin Cyclides on the 3-Sphere

This example ray traces Dupin cyclides inside the curved 3-sphere, where light follows great circles all the way around to the antipode and back to the eye. A spherical cylinder surrounds a great circle at constant angular radius, and a conformal dilation toward an off-core aim bends that cylinder into symmetric tori and lopsided Dupin cyclides while closing a two-vertex spherical cone into a spindle cyclide.

In conformal geometric algebra, points of the 3-sphere are modelled as null antivectors, spheres as vectors, and quadrics as polarity maps from points to their polar spheres built from dyads `sphere * (sphere & Point)`. Rotations, conformal dilations, and sphere inversions transform a quadric on both its output sphere and its input point via `motion >> quadric(motion << Point)`. Reparameterizing each great-circle ray by the half-angle tangent turns the ray into a point parabola whose substitution into `Point & surfaces` assembles five multilinear direction maps up to `form(ray_bend, ray_bend)` once before evaluating pixel directions.

In Cartesian and Möbius coordinate notation, a ring or spindle Dupin cyclide in $\mathbb{R}^3$ satisfies the implicit quartic $(x^2 + y^2 + z^2 + b^2 - d^2)^2 - 4(ax - cd)^2 - 4b^2 y^2 = 0$, while stereographic lifts $\pi^{-1}(\mathbf{p}) = (2\mathbf{p},\, 1 - \|\mathbf{p}\|^2)/(1 + \|\mathbf{p}\|^2)$ and Euclidean sphere inversions $\mathbf{p} \mapsto \mathbf{c} + R^2(\mathbf{p} - \mathbf{c})/\|\mathbf{p} - \mathbf{c}\|^2$ are composed separately from orthogonal matrices in $\mathrm{SO}(4)$. Tracing a great-circle arc $\mathbf{x}(t) = \mathbf{x}_0 \cos t + \mathbf{d} \sin t$ against a homogeneous quadratic $\mathbf{x}^\top Q \mathbf{x} = 0$ then requires Weierstrass substitution $u = \tan(t/2)$ and explicit rank-four index contractions $Q_{ijkl}\, d^i d^j d^k d^l$ at each pixel.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator

import numpy as np

from numga import Algebra, NumpyContext, concatenate, stack
from numga.algebras import CGA3D
from examples.util.animation import save_figure, show_states
from examples.quadrics.cyclides import render, scenarios
from examples.quadrics.cyclides.numerical import nearest_hit_angles, pixel_offsets

ga = CGA3D
ctx = NumpyContext(ga)
mv = ctx.multivector

Scalar = ga.gatype.scalar()                                                    # Scalar+
# Spheres and great spheres are vectors, and points of the 3-sphere are null antivectors.
Sphere = ga.gatype.vector()                                                    # Vector+
Point = ga.gatype.antivector()                                                 # Antivector-
Upright = Sphere.derive.within(Algebra("x+y+z+"))                              # Vector+
Direction = Upright.derive.dual                                                # Antivector-
Motor = ga.gatype.rotor()                                                      # Rotor+
Bivector = ga.gatype.bivector()                                                # Bivector+
Quadric = ga.gatype((Sphere, Point))                                           # Sphere+ <- Point-

# The eye sits at w on the 3-sphere and its antipode sits opposite at -w.
origin = mv.w.dual() + mv.e.dual()                                             # [] Point-
antipode = mv.e.dual() - mv.w.dual()                                           # [] Point-

np.set_printoptions(precision=6, suppress=True)

## Points and spheres

Each point of the 3-sphere adds its unit four-space direction to one unit of the ambient centre, forming a null antivector whose regressive product with the ambient plane equals minus one everywhere on the 3-sphere. Adding a cosine multiple of that ambient plane to a great sphere shifts the zero set of `sphere & point` from a quarter turn to a chosen angular radius around the eye.

In [ ]:
radius = 0.4

In [ ]:
# Offsetting the great sphere w by cos(radius) along e places the zero set at angular radius 0.4 from the eye.
test_sphere = mv.w + mv.e * np.cos(radius)                                     # [] Sphere+
eye_turn = (mv.xw * (radius / 2)).exp()                                        # [] Motor+
turned_eye = eye_turn >> origin                                                # [] Point-
sphere_readings = stack((
    test_sphere & origin,
    mv.e & origin,
    mv.e & turned_eye,
))                                                                             # [3] Scalar+

print("sphere & origin, e & origin, e & turned_eye:", sphere_readings.to_array())

## Rays as great circles

Turning the eye toward a unit tangent direction traces a great circle whose generator squares to minus one, so repeated commutators give the ray's initial velocity and acceleration as multilinear maps in that direction. Rescaling the turned point by one plus the squared half-angle tangent clears trigonometric denominators without moving the projective point and turns the great circle into a parabola whose quadratic bend is the antipode of the eye.

In [ ]:
# Joining w with a tangent direction d spans the four-space rotation plane whose dual generates the great-circle ray.
ray_rotation = 0.5 * (mv.w.dual() & Direction).dual()                          # [] Bivector+ <- Direction-
ray_linear = ray_rotation.commutator(origin) * 2                               # [] Point- <- Direction-
ray_quadratic = ray_rotation.commutator(ray_linear) * 2                        # [] Point- <- (Direction-, Direction-)
# Clearing 1 + u**2 in u = tan(t / 2) turns the great circle into a parabola whose u**2 bend is the antipode.
ray_bend = ray_quadratic * 2 + origin * Direction.dot(Direction)               # [] Point- <- (Direction-, Direction-)

print(ray_quadratic)

## Tracing the spherical cylinder

A quadric built from sphere dyads `sphere * (sphere & Point)` maps each point to its polar sphere, and a point lies on the surface when it is incident with its own polar. Combining two orthogonal great spheres with the ambient plane forms a spherical cylinder of constant angular radius around a great circle.

Substituting the parabolic ray into `Point & surfaces` produces five multilinear direction maps for the constant through quartic powers of the half-angle tangent before any pixel direction is bound. At each hit, the polar sphere is tangent to the surface and its reverse norm scales with the surface gradient, giving the headlight cosine directly from `polars & velocities`.

In [ ]:
panel_shape = scenarios.panel_shape
cylinder_tube = scenarios.cylinder_tube

In [ ]:
def cylinder_quadrics(tubes: np.ndarray) -> Quadric:
    """Points at angular radius `tubes` from the `xy` great circle satisfy a spherical-cylinder
    polarity map (`[...] Sphere+ <- Point-`)."""
    return mv.z * (mv.z & Point) + mv.w * (mv.w & Point) - np.sin(tubes) ** 2 * mv.e * (mv.e & Point)


cylinder = cylinder_quadrics(cylinder_tube)                                    # [] Sphere+ <- Point-
toward_view = (mv.xw * ((np.pi / 2 - 1.2) / 2)).exp()                          # [] Motor+
core_turn = (mv.yz * ((np.pi / 2 - 0.3) / 2)).exp()                            # [] Motor+
cylinder_pose = toward_view * core_turn                                        # [] Motor+
placed_cylinder = (cylinder_pose >> cylinder(cylinder_pose << Point)).reshape(1)  # [1] Sphere+ <- Point-

In [ ]:
def sensor_directions(image_shape: tuple[int, int], fov: float) -> Direction:
    """Unit pixel directions (`[...] Direction-`) at the eye look along `-x` with `z` up in
    row-major order."""
    right, up = pixel_offsets(image_shape, fov)
    rays = -mv.x.dual() + mv.y.dual() * right + mv.z.dual() * up               # [...] Direction-
    return rays / rays.antinorm()


def surface_trace(surfaces: Quadric, pixels: Direction) -> tuple[Scalar, Scalar]:
    """Tracing `surfaces` along unit `pixels` gives the headlight cosine and nearest hit angle
    (`[...] Scalar+`, `[...] Scalar+`), with `inf` on a miss."""
    form = Point & surfaces                                                    # [...] Scalar+ <- (Point-, Point-)

    # Evaluating form(X, X) along the parabolic ray collects five direction maps, one per power of u = tan(t / 2).
    constant = form(origin, origin)                                            # [...] Scalar+
    linear = 4 * form(origin, ray_linear)                                      # [...] Scalar+ <- Direction-
    quadratic = 4 * form(ray_linear, ray_linear) + 2 * form(origin, ray_bend)  # [...] Scalar+ <- (Direction-, Direction-)
    cubic = 4 * form(ray_linear, ray_bend)                                     # [...] Scalar+ <- (Direction-, Direction-, Direction-)
    quartic = form(ray_bend, ray_bend)                                         # [...] Scalar+ <- (Direction-, Direction-, Direction-, Direction-)

    angles = nearest_hit_angles([
        constant[:, None],
        linear[:, None](pixels),
        quadratic[:, None](pixels, pixels),
        cubic[:, None](pixels, pixels, pixels),
        quartic[:, None](pixels, pixels, pixels, pixels),
    ])                                                                         # [...] Scalar+
    clamped = angles.clip(0.0, 2 * np.pi)                                      # [...] Scalar+
    hits = (origin + ray_linear(pixels) * clamped.sin()
            + ray_quadratic(pixels, pixels) * (1 - clamped.cos()))             # [...] Point-
    velocities = (ray_linear(pixels) * clamped.cos()
                  + ray_quadratic(pixels, pixels) * clamped.sin())             # [...] Point-

    # The polar sphere at a surface point is tangent there, and its reverse norm scales with the surface gradient.
    polars = surfaces[:, None](hits)                                           # [...] Sphere+
    facings = -(polars & velocities) / (polars | polars).abs().square_root()   # [...] Scalar+
    return facings, angles


pixels = sensor_directions(panel_shape, np.radians(90))                        # [...] Direction-
cylinder_facing, cylinder_angle = surface_trace(placed_cylinder, pixels)       # [1, ...] Scalar+, [1, ...] Scalar+
save_figure(render.facing_figure(cylinder_facing, cylinder_angle, panel_shape), "cyclides_cylinder");

Teal headlight shading brightens where the surface normal faces the eye along the ray and darkens toward the grazing silhouettes against the background. Because great circles of the 3-sphere curve across a wide pinhole view, the straight core of the spherical cylinder arches smoothly from top to bottom.

## Tori from a bent core

A conformal dilation fixes an aim point and its antipode while sliding every other point along the great circle joining them, generated by the dual of the four-space diameter through the ambient centre and the aim. Aiming that dilation a quarter turn from every point of the cylinder's core pulls the great-circle core evenly into a smaller circle and turns the spherical cylinder into a symmetric torus whose cross-sectional ratio is set by the initial tube radius.

In [ ]:
torus_tubes = scenarios.torus_tubes
torus_strengths = scenarios.torus_strengths

In [ ]:
def placement_motors(toward: Point, ahead: np.ndarray, tilt: np.ndarray) -> Motor:
    """Motors (`[...] Motor+`) tilt about the `yw` plane and carry `toward` onto the point `ahead`
    radians along `-x` in front of the eye."""
    ahead_turn = (mv.wx * (ahead / 2)).exp()                                   # [...] Motor+
    target = ahead_turn >> mv.w.dual()                                         # [...] Point-
    onto_target = (1 + target * toward.inverse()).normalized()                 # [...] Motor+
    tilting = (mv.yw * (tilt / 2)).exp()                                       # [...] Motor+
    return onto_target * tilting


torus_dilations = (mv.ze * (torus_strengths / 2)).exp()                        # [3] Motor+
torus_view = placement_motors(mv.z.dual(), 1.1, 0.8)                           # [] Motor+
torus_poses = torus_view * torus_dilations                                     # [3] Motor+
torus_cylinders = cylinder_quadrics(torus_tubes)                               # [3] Sphere+ <- Point-
placed_tori = torus_poses >> torus_cylinders(torus_poses << Point)             # [3] Sphere+ <- Point-

tori_facings, tori_angles = surface_trace(placed_tori, pixels)                 # [3, ...] Scalar+, [3, ...] Scalar+
save_figure(render.facing_figure(tori_facings, tori_angles, panel_shape), "cyclides_tori");

From left to right, widening the initial cylinder from 0.15 to 0.9 radians while strengthening the dilation to fill the frame carries a slender ring into a thick torus with a narrow central hole.

## Lopsided Dupin cyclides

Leaning the dilation aim off the symmetry axis toward a point on the core breaks rotational symmetry around the ring, compressing the tube on the side nearest the aim and swelling the opposite side into a lopsided Dupin cyclide.

In [ ]:
dupin_tubes = scenarios.dupin_tubes
dupin_strengths = scenarios.dupin_strengths
dupin_leans = scenarios.dupin_leans
dupin_ahead = scenarios.dupin_ahead
dupin_tilts = scenarios.dupin_tilts

In [ ]:
dupin_aim_turns = (mv.zx * (-dupin_leans / 2)).exp()                           # [3] Motor+
dupin_aims = dupin_aim_turns >> mv.z.dual()                                    # [3] Point-
dupin_dilations = ((mv.e.dual() & dupin_aims).dual() * (dupin_strengths / 2)).exp()  # [3] Motor+
dupin_cylinders = cylinder_quadrics(dupin_tubes)                               # [3] Sphere+ <- Point-
lopsided_cyclides = dupin_dilations >> dupin_cylinders(dupin_dilations << Point)  # [3] Sphere+ <- Point-
dupin_views = placement_motors(dupin_aims, dupin_ahead, dupin_tilts)           # [3] Motor+
placed_dupin = dupin_views >> lopsided_cyclides(dupin_views << Point)          # [3] Sphere+ <- Point-

dupin_facings, dupin_angles = surface_trace(placed_dupin, pixels)              # [3, ...] Scalar+, [3, ...] Scalar+
save_figure(render.facing_figure(dupin_facings, dupin_angles, panel_shape), "cyclides_dupin");

Each cyclide narrows on the side closest to the off-axis dilation aim and swells opposite it, viewed from three inclinations around the ring.

## Spindles from a cone

Geodesics leaving a vertex at a fixed angle from an axis reconverge at the antipodal vertex, giving a spherical cone with two conical points built from two great spheres and the ambient plane. A strong dilation aimed toward the axis pulls those two vertices together so the cone closes into a spindle cyclide with nested outer and inner sheets, while leaning the aim off the axis curves the inner sheet behind a side opening.

In [ ]:
cone_opening = scenarios.cone_opening
spindle_leans = scenarios.spindle_leans
spindle_shots = scenarios.spindle_shots

In [ ]:
def cone_quadric(opening: float) -> Quadric:
    """Geodesics leaving `z` and `-z` at half-opening `opening` around the `x` axis form a
    two-vertex spherical cone (`[] Sphere+ <- Point-`)."""
    return mv.x * (mv.x & Point) + np.cos(opening) ** 2 * (mv.z * (mv.z & Point) - mv.e * (mv.e & Point))


spindle_aim_turns = (mv.xy * (-spindle_leans / 2)).exp()                       # [2] Motor+
spindle_aims = spindle_aim_turns >> mv.x.dual()                                # [2] Point-
spindle_dilations = ((mv.e.dual() & spindle_aims).dual() * (3.0 / 2)).exp()    # [2] Motor+
cone = cone_quadric(cone_opening)                                              # [] Sphere+ <- Point-
spindles = spindle_dilations >> cone(spindle_dilations << Point)               # [2] Sphere+ <- Point-
spindle_midpoints = -spindle_aims                                              # [2] Point-

In [ ]:
def coordinate_rotation(angles: list[float]) -> Motor:
    """Six coordinate-plane angles in `xy`, `xz`, `xw`, `yz`, `yw`, and `zw` compose into a
    four-dimensional rotation (`[] Motor+`)."""
    rotor = mv.rotor()                                                         # [] Motor+
    for plane, angle in zip((mv.xy, mv.xz, mv.xw, mv.yz, mv.yw, mv.zw), angles):
        step = (plane * angle).exp()                                           # [] Motor+
        rotor = rotor * step                                                   # [] Motor+
    return rotor


def camera_view_motor(centre: Point, angles: list[float], ahead: float, yaw: float, pitch: float) -> Motor:
    """Rotating by `angles`, carrying the rotated `centre` to `ahead` radians in front of the eye,
    and yawing and pitching at `w` gives a view motor (`[] Motor+`)."""
    body_turn = coordinate_rotation(angles)                                    # [] Motor+
    ahead_turn = (mv.wx * (ahead / 2)).exp()                                   # [] Motor+
    target = ahead_turn >> mv.w.dual()                                         # [] Point-
    rotated_centre = body_turn >> centre                                       # [] Point-
    carry = (1 + target * rotated_centre.inverse()).normalized()               # [] Motor+
    yawing = (mv.xy * (yaw / 2)).exp()                                         # [] Motor+
    pitching = (mv.xz * (pitch / 2)).exp()                                     # [] Motor+
    return yawing * pitching * carry * body_turn


spindle_views = stack([
    camera_view_motor(spindle_midpoints[i], angles, ahead, yaw, pitch)
    for i, angles, ahead, yaw, pitch in spindle_shots
])                                                                             # [5] Motor+
spindle_surfaces = stack([spindles[i] for i, *_ in spindle_shots])             # [5] Sphere+ <- Point-
placed_spindles = spindle_views >> spindle_surfaces(spindle_views << Point)    # [5] Sphere+ <- Point-

raw_spindle_facings, spindle_angles = surface_trace(placed_spindles, pixels)   # [5, ...] Scalar+, [5, ...] Scalar+
spindle_facings = raw_spindle_facings.abs()                                    # [5, ...] Scalar+
save_figure(render.facing_figure(spindle_facings, spindle_angles, panel_shape), "cyclides_spindles");

The first three views show the symmetric spindle cyclide and its two conical vertices, while the final two peer through the side opening of the leaning dilation at the curved inner sheet shaded on both faces.

## Palindromic quartic of the cylinder

Because the spherical cylinder combines great spheres with the ambient plane alone, antipodal points on the 3-sphere evaluate identically in its quadratic form and every great-circle ray meets it in antipodal hit pairs. Along the parabolic ray, its quartic and constant coefficients agree while its cubic and linear coefficients have opposite signs, whereas an off-axis conformal dilation breaks both symmetries.

In [ ]:
comparison_surfaces = stack((placed_cylinder[0], placed_dupin[0]))             # [2] Sphere+ <- Point-
comparison_forms = Point & comparison_surfaces                                 # [2] Scalar+ <- (Point-, Point-)
sample_pixels = pixels[::20000]                                                # [...] Direction-
single_ray = sample_pixels[0]                                                  # [] Direction-

ray_coefficients = stack((
    comparison_forms(origin, origin),
    4 * comparison_forms(origin, ray_linear)(single_ray),
    (4 * comparison_forms(ray_linear, ray_linear) + 2 * comparison_forms(origin, ray_bend))(single_ray, single_ray),
    4 * comparison_forms(ray_linear, ray_bend)(single_ray, single_ray, single_ray),
    comparison_forms(ray_bend, ray_bend)(single_ray, single_ray, single_ray, single_ray),
), axis=-1)                                                                    # [2, 5] Scalar+

cylinder_dupin_outer = (
    comparison_forms(ray_bend, ray_bend)[:, None](sample_pixels, sample_pixels, sample_pixels, sample_pixels)
    - comparison_forms(origin, origin)[:, None]
)                                                                              # [2, ...] Scalar+
cylinder_dupin_odd = (
    (4 * comparison_forms(ray_linear, ray_bend))[:, None](sample_pixels, sample_pixels, sample_pixels)
    + (4 * comparison_forms(origin, ray_linear))[:, None](sample_pixels)
)                                                                              # [2, ...] Scalar+

print("constant through quartic coefficients for cylinder vs Dupin cyclide:\n", ray_coefficients.to_array())

## Vortex flow around a tilted circle

Meeting two spheres by the exterior product forms a circle bivector whose exponential carries points of the 3-sphere around that circle like fluid circulating around a vortex ring. Flowing around the torus's own core circle spins the surface in place, whereas tilting the circle slightly makes one turn roll and deform the torus through a family of lopsided cyclides. Tracing the 54 frames takes a few seconds.

In [ ]:
n_frames = scenarios.n_frames
frame_duration_ms = scenarios.frame_duration_ms

In [ ]:
def vortex_states(torus_surface: Quadric, circle: Bivector, pixels: Direction,
                  n_frames: int) -> Iterator[tuple[Scalar, Scalar]]:
    """Yields `(facings, angles)` (`[1, ...] Scalar+`) as `torus_surface` flows once around unit
    bivector `circle` over `n_frames` frames."""
    phases = np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False)
    flows = (circle * (phases / 2)).exp()                                      # [n_frames] Motor+
    for surface in flows >> torus_surface(flows << Point):
        yield surface_trace(surface.reshape(1), pixels)


vortex_tilt = (mv.yw * 0.2).exp()                                              # [] Motor+
vortex_motor = torus_poses[1] * vortex_tilt                                    # [] Motor+
vortex_circle = vortex_motor >> mv.zw                                          # [] Bivector+

vortex_records = show_states(
    vortex_states(placed_tori[1], vortex_circle, pixels, n_frames),
    render.facing_animation, panel_shape,
    name="cyclides_vortex", duration_ms=frame_duration_ms, n_frames=n_frames,
)

Over one period around the tilted circle, the symmetric torus rocks into a lopsided Dupin cyclide as one side passes closer to the vortex core and returns smoothly to its initial ring.

## Linked vortex in the flat chart

Point spheres of zero radius at the eye and its antipode act as the origin and point at infinity of the stereographic flat chart, so flat conformal constructions draw identical surfaces directly on the 3-sphere. Inverting a one-sheet hyperboloid in an exterior sphere sends infinity to the sphere's centre and closes both open ends into one conical tip, while the inverted waist circle threads the opening and drives a linked vortex motion around a thin stationary ring. Tracing the larger 54-frame scene takes around ten seconds.

In [ ]:
linked_shape = scenarios.linked_shape
ring_radius, ring_tube = 2.5, 0.02
camera_position = mv(Upright, [2.8, -9.0, 4.8]).dual()                         # [] Direction-
camera_target = mv(Upright, [1, 0, 0]).dual()                                  # [] Direction-

In [ ]:
# Point spheres at the eye and its antipode serve as origin and infinity of the stereographic flat chart.
chart_origin = (mv.w + mv.e) * 0.5                                             # [] Sphere+
chart_infinity = mv.e - mv.w                                                   # [] Sphere+

hyperboloid = (mv.z * (mv.z & Point) - mv.x * (mv.x & Point) - mv.y * (mv.y & Point) / 0.65 ** 2
               + chart_infinity * (chart_infinity & Point))                    # [] Sphere+ <- Point-
inversion_shift = ((antipode & (mv.x.dual() * 1.7)).dual() * (-1 / 2)).exp()   # [] Motor+
shifted_sphere = inversion_shift >> (chart_origin - chart_infinity)            # [] Sphere+
inversion = shifted_sphere.normalized()                                        # [] Sphere+
linked_cyclide = -(inversion >> hyperboloid(inversion << Point))               # [] Sphere+ <- Point-

waist_sphere = chart_origin - chart_infinity * (ring_radius ** 2 / 2)          # [] Sphere+
inverted_circle = inversion >> (mv.z ^ waist_sphere)                           # [] Bivector+
linked_circle = inverted_circle.normalized()                                   # [] Bivector+

inverted_sphere = inversion >> waist_sphere                                    # [] Sphere+
unit_sphere = inverted_sphere / -(inverted_sphere | chart_infinity)            # [] Sphere+
radius_squared = unit_sphere.squared()                                         # [] Scalar+
ring_sphere = unit_sphere - chart_infinity * (ring_tube ** 2 / 2)              # [] Sphere+
linked_ring = ring_sphere * (ring_sphere & Point) + radius_squared * (
    mv.z * (mv.z & Point) - ring_tube ** 2 * chart_infinity * (chart_infinity & Point)
)                                                                              # [] Sphere+ <- Point-

In [ ]:
def flat_camera_motor(position: Direction, target: Direction) -> Motor:
    """Inverting the pose of a pinhole camera at `position` aimed at `target` brings the flat
    chart's camera onto the eye at `w` looking along `-x` (`[] Motor+`)."""
    sight = target - position                                                  # [] Direction-
    forward = sight / sight.antinorm()                                         # [] Direction-
    upright = mv.z.dual() - (mv.z.dual() | forward) * forward.inverse()        # [] Direction-
    up = upright / upright.antinorm()                                          # [] Direction-
    aim = (1 + forward * (-mv.x.dual()).inverse()).normalized()                # [] Motor+
    aimed_up = aim >> mv.z.dual()                                              # [] Direction-
    roll = (1 + up * aimed_up.inverse()).normalized()                          # [] Motor+
    translation = ((antipode & position).dual() * (-1 / 2)).exp()              # [] Motor+
    return (translation * roll * aim).inverse()


def linked_vortex_states(cyclide_surface: Quadric, ring_surface: Quadric, circle: Bivector,
                         camera: Motor, pixels: Direction, n_frames: int) -> Iterator[tuple[Scalar, Scalar]]:
    """Yields `(facings, angles)` (`[2, ...] Scalar+`) as `cyclide_surface` flows once around unit
    bivector `circle` linked with static `ring_surface` over `n_frames` frames."""
    placed_ring = (camera >> ring_surface(camera << Point)).reshape(1)         # [1] Sphere+ <- Point-
    ring_facing, ring_angle = surface_trace(placed_ring, pixels)               # [1, ...] Scalar+, [1, ...] Scalar+
    phases = np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False)
    flows = (circle * (phases / 2)).exp()                                      # [n_frames] Motor+
    for motion in camera * flows:
        placed_cyclide = (motion >> cyclide_surface(motion << Point)).reshape(1)  # [1] Sphere+ <- Point-
        facing, angle = surface_trace(placed_cyclide, pixels)                  # [1, ...] Scalar+, [1, ...] Scalar+
        yield concatenate([facing, ring_facing]).abs(), concatenate([angle, ring_angle])


linked_camera = flat_camera_motor(camera_position, camera_target)              # [] Motor+
linked_pixels = sensor_directions(linked_shape, np.radians(44))                # [...] Direction-

linked_records = show_states(
    linked_vortex_states(linked_cyclide, linked_ring, linked_circle, linked_camera, linked_pixels, n_frames),
    render.linked_animation, linked_shape,
    name="cyclides_linked_vortex", duration_ms=frame_duration_ms, n_frames=n_frames,
)

The teal cyclide turns continuously around the gold ring, sweeping its singular conical tip through the opening and back around the outside without intersecting the ring.

## Gallery of flat-chart scenes

Additional quadrics in `scenarios.py` combine coordinate planes, point spheres, and sphere inversions in the flat chart around the eye, traced through the same spherical ray polynomial as static stills and linked vortex animations. Tracing both vortex animations takes around ten seconds.

In [ ]:
gallery_images = [
    next(render.scene_animation(panel_shape, render.body_palette, scenarios.flat_scene_states(name, 1)))
    for name in scenarios.still_scene_names
]
save_figure(render.gallery_figure(gallery_images, 4), "cyclides_gallery");

In [ ]:
gallery_vortex_records = [
    show_states(
        scenarios.flat_scene_states(name, n_frames),
        render.scene_animation, panel_shape, render.body_palette,
        name=f"cyclides_{name}", duration_ms=frame_duration_ms, n_frames=n_frames,
    )
    for name in scenarios.vortex_scene_names
]

The eight stills survey ring, peanut, pinched, hyperboloidal, and four-handle quadrics in the flat chart, followed by a tilted torus and a pinched cyclide each orbiting a linked horizontal ring.

In [ ]:
# checks
check_direction = mv.y.dual() * 0.6 + mv.z.dual() * 0.8                        # [] Direction-
check_angles = np.array([0.7, 2.5])
check_rotors = (ray_rotation(check_direction) * check_angles).exp()            # [2] Motor+
exact_points = check_rotors >> origin                                          # [2] Point-
circle_points = (origin + ray_linear(check_direction) * np.sin(check_angles)
                 + ray_quadratic(check_direction, check_direction) * (1 - np.cos(check_angles)))  # [2] Point-
radius_spheres = mv.w + mv.e * np.cos(check_angles)                            # [2] Sphere+
half_tangents = np.tan(check_angles / 2)
parabola_points = (origin + ray_linear(check_direction) * (2 * half_tangents)
                   + ray_bend(check_direction, check_direction) * half_tangents ** 2)  # [2] Point-
basis_probes = stack((mv.x, mv.y, mv.z, mv.w, mv.e))                           # [5] Sphere+

assert ((test_sphere & turned_eye).abs() < 1e-10).all()
assert (((mv.e & turned_eye) + 1).abs() < 1e-10).all()
assert ((radius_spheres & exact_points).abs() < 1e-7).all()
assert ((radius_spheres & circle_points).abs() < 1e-10).all()
assert ((basis_probes & (ray_bend(check_direction, check_direction) - antipode)).abs() < 1e-10).all()
assert ((basis_probes[:, None] & (parabola_points / (1 + half_tangents ** 2) - circle_points)).abs() < 1e-10).all()
assert cylinder_angle.isfinite().any()
assert tori_angles.isfinite().any(axis=-1).all()
assert dupin_angles.isfinite().any(axis=-1).all()
assert spindle_angles.isfinite().any(axis=-1).all()
assert (cylinder_dupin_outer[0].abs() < 1e-10).all()
assert (cylinder_dupin_odd[0].abs() < 1e-10).all()
assert (cylinder_dupin_outer[1].abs() > 1e-3).all()
assert (cylinder_dupin_odd[1].abs() > 1e-3).all()
assert (((vortex_circle | vortex_circle) + 1).abs() < 1e-10).all()
assert len(vortex_records) == n_frames and vortex_records[0][1].isfinite().any()
assert (((linked_circle | linked_circle) + 1).abs() < 1e-10).all()
assert len(linked_records) == n_frames and linked_records[0][1].isfinite().any(axis=-1).all()
assert len(gallery_images) == len(scenarios.still_scene_names)
assert all(len(records) == n_frames and records[0][1].isfinite().any() for records in gallery_vortex_records)